# Kaggle Models Metadata Extractor — Complete Tutorial

**Goal:** Extract complete model card metadata from every publicly listed model on Kaggle and save it as a structured JSON file.

**Author:** [Your Name]  
**Date:** [Date]  

---

## Key Concepts

### What is a Model Card?
A model card is a structured document that describes a machine learning model. It includes the model's purpose, architecture, training data, usage examples, licensing, and all available variations (called "instances"). On Kaggle, every model has a public model card page — this notebook extracts that information programmatically via the Kaggle API.

### What is a `ref`?
A `ref` (reference) is the **unique identifier** for a model on Kaggle. It follows the format `owner/model-name`:

| ref | Owner | Model | URL |
|-----|-------|-------|-----|
| `google/bert` | Google | BERT | kaggle.com/models/**google/bert** |
| `tensorflow/efficientnet` | TensorFlow | EfficientNet | kaggle.com/models/**tensorflow/efficientnet** |
| `meta-llama/llama-3` | Meta | Llama 3 | kaggle.com/models/**meta-llama/llama-3** |

### What is an Instance?
A single model can have multiple **instances** (variations). These are different deployments of the same model for different frameworks or configurations. For example, `google/bert` has instances like:
- `answer-equivalence-bem` (TensorFlow2)
- `bert-uncased-l-12-h-768-a-12` (TensorFlow1)
- Same model weights repackaged for PyTorch

Each instance has its own framework, usage docs, version history, license, and file size.

### Two-Phase Workflow

| Phase | API Method | What it does | Output |
|-------|-----------|-------------|--------|
| **Phase 1** | `model_list` | Gets list of all model refs (basic info only) | ~6,250 refs |
| **Phase 2** | `model_get(ref)` | For each ref, fetches the FULL model card | Complete metadata |

### Output JSON Structure
The final output is a JSON array. Each element is a complete model card:
```json
[
    {
        "id": 113,
        "ref": "google/bert",
        "title": "bert",
        "subtitle": "BERT model trained on...",
        "author": "Google",
        "slug": "bert",
        "description": "Full markdown description...",
        "url": "https://www.kaggle.com/models/google/bert",
        "tags": [...],
        "publish_time": "2023-01-11T19:46:42Z",
        "update_time": "2024-03-15T10:22:00Z",
        "vote_count": 236,
        "provenance_sources": "...",
        "instances": [
            {
                "id": 934,
                "slug": "answer-equivalence-bem",
                "framework": "MODEL_FRAMEWORK_TENSOR_FLOW_2",
                "fineTunable": true,
                "overview": "Short description...",
                "usage": "Detailed usage in markdown...",
                "versionId": 1085,
                "versionNumber": 1,
                "trainingData": ["other"],
                "licenseName": "Apache 2.0",
                "totalUncompressedBytes": 442081280
            }
        ]
    }
]
```


---
## Step 1 - Install Dependencies & Set API Token

### How to get your Kaggle API Token:
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Under **"API"**, click **"Generate New Token"**
3. Copy the token string
4. Paste it below where it says `PASTE_YOUR_TOKEN_HERE`


In [ ]:
# Install the official Kaggle package
!pip install kaggle -q

In [ ]:
import os

# ⬇️ PASTE YOUR KAGGLE API TOKEN HERE ⬇️
os.environ["KAGGLE_API_TOKEN"] = ""


---
## Step 2 - Authenticate & Test Connection

This verifies your token works by fetching one model's metadata.


In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# Quick test — fetch one model to confirm everything works
test = api.model_get("google/bert")
print(f"Authentication successful!")
print(f"Test model : {test.title} by {test.author}")
print(f"Instances  : {len(test.instances)}")
print(f"Description: {test.description[:150]}...")


---
## Step 3 - Phase 1: Fetch All Model Refs

This calls the `model_list` API to get the list of every publicly available model on Kaggle. It returns basic info (ref, title, author) — we'll enrich this with full model cards in Phase 2.

**Note:** Kaggle's API has a server-side pagination limit of ~6,250 models. The function handles this gracefully with retry logic and saves progress automatically.

**Expected time:** ~5 minutes


In [ ]:
import time
import json
from kagglesdk.models.types.model_api_service import ApiListModelsRequest


def fetch_model_refs(output_file="kaggle_models_metadata.json",
                     max_pages=500, max_retries=3):
    """
    Phase 1: Fetches the list of all model refs from Kaggle.
    
    Calls the model_list API endpoint repeatedly, paginating through
    all available models. Each page returns up to 50 models with basic
    info (ref, title, author, vote count).
    
    Parameters
    ----------
    output_file : str
        Where to save the model list as JSON.
    max_pages : int
        Safety cap on pagination (default 500).
    max_retries : int
        Retry attempts per page on server errors.
    
    Returns
    -------
    list[dict]
        List of model dictionaries with basic metadata.
    """
    api = KaggleApi()
    api.authenticate()

    all_models = []
    page_token = None
    page = 0

    while page < max_pages:
        page += 1
        print(f"  Page {page} ...", end=" ", flush=True)

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                with api.build_kaggle_client() as kaggle:
                    request = ApiListModelsRequest()
                    request.search = ""
                    request.owner = ""
                    request.page_size = 50
                    if page_token:
                        request.page_token = page_token
                    response = kaggle.models.model_api_client.list_models(request)
                success = True
                break
            except Exception as e:
                wait = 10 * attempt
                print(f"\n    Attempt {attempt}/{max_retries} failed: {type(e).__name__}")
                if attempt < max_retries:
                    print(f"    Retrying in {wait}s ...", end=" ", flush=True)
                    time.sleep(wait)

        if not success:
            print(f"\n  Stopped after {max_retries} retries. Saving what we have.")
            break

        models = response.models or []
        if not models:
            print("no more results.")
            break

        # Convert model objects to dicts
        for m in models:
            all_models.append(m.to_dict() if hasattr(m, "to_dict") else str(m))
        print(f"got {len(models)}  (total: {len(all_models)})")

        if len(models) < 50:
            break

        page_token = response.next_page_token
        if not page_token:
            print("  No more pages.")
            break

        time.sleep(1.5)

    # Save to JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_models, f, indent=2, ensure_ascii=False, default=str)

    print(f"\n  Done — {len(all_models)} models saved to {output_file}")
    return all_models


In [ ]:
# Run Phase 1
model_refs = fetch_model_refs()


---
## Step 4 - Phase 2: Define the Model Card Fetcher

This is the main function that fetches the **full model card** for every model. It:
- Reads the refs from Phase 1
- Calls `model_get(ref)` for each one to get the complete metadata
- Saves everything to a single JSON file
- Is **fully resumable** — if interrupted, re-run and it picks up where it left off
- Saves progress every 50 models so nothing is lost on a crash


In [ ]:
import os


def fetch_all_model_cards(
    models_json="kaggle_models_metadata.json",
    output_file="kaggle_models_full_cards_now.json",
    delay=2,
    max_retries=3,
    limit=None
):
    """
    Phase 2: Fetches the complete model card for every model and saves
    all results to a single output JSON file.

    This function reads model refs from the Phase 1 output, then calls
    the Kaggle API's model_get endpoint for each ref to retrieve the
    full model card — including description, instances, frameworks,
    tags, licenses, provenance, training data, and usage documentation.

    Parameters
    ----------
    models_json : str, default "kaggle_models_metadata.json"
        Path to the input JSON file from Phase 1. Each entry must have
        a "ref" field (e.g., "google/bert").

    output_file : str, default "kaggle_models_full_cards.json"
        Path where the enriched model cards will be saved as a JSON array.

    delay : int or float, default 2
        Seconds to wait between API calls. Prevents hitting Kaggle's
        rate limits. Recommended: 2 seconds (~1,800 calls/hour).

    max_retries : int, default 3
        Retry attempts per model if the API call fails. Each retry waits
        progressively longer: 5s, 10s, 15s.

    limit : int or None, default None
        If set, only fetches the first N models. Useful for testing.
        Set to None to fetch all models.

    Returns
    -------
    dict
        A dictionary mapping each model ref (str) to its full model
        card (dict).

    Resumability
    ------------
    Fully resumable. If interrupted, re-run the same call — it loads
    the output file, skips already-fetched models, and continues.
    Progress is saved every 50 models.

    Examples
    --------
    >>> enriched = fetch_all_model_cards(limit=10)   # test run
    >>> enriched = fetch_all_model_cards()            # full run
    >>> enriched = fetch_all_model_cards()            # resume
    """
    # ── 1. Authenticate
    api = KaggleApi()
    api.authenticate()

    # ── 2. Load model refs from Phase 1
    with open(models_json, "r", encoding="utf-8") as f:
        models = json.load(f)

    refs = [m.get("ref", "") for m in models if m.get("ref")]
    refs = list(dict.fromkeys(refs))  # deduplicate, keep order
    if limit:
        refs = refs[:limit]

    # ── 3. Load previous progress (for resumability)
    enriched = {}
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            existing = json.load(f)
            for m in existing:
                enriched[m.get("ref", "")] = m
        print(f"  Resuming — {len(enriched)} already done, "
              f"{len(refs) - len(enriched)} remaining")
    except FileNotFoundError:
        print(f"  Starting fresh")

    print(f"  Total models: {len(refs)}\n")

    # ── 4. Fetch full model card for each ref
    new_count = 0
    failed_refs = []

    for i, ref in enumerate(refs):
        # Skip if already fetched
        if ref in enriched:
            continue

        print(f"  [{i+1}/{len(refs)}] {ref} ...", end=" ", flush=True)

        # Retry loop with backoff
        success = False
        for attempt in range(1, max_retries + 1):
            try:
                detail = api.model_get(ref)
                enriched[ref] = detail.to_dict()
                print("OK")
                success = True
                new_count += 1
                break
            except Exception as e:
                if attempt < max_retries:
                    wait = 5 * attempt
                    print(f"retry {attempt} ...", end=" ", flush=True)
                    time.sleep(wait)
                else:
                    print(f"FAILED ({type(e).__name__})")
                    failed_refs.append(ref)

        # Save progress every 50 new models
        if new_count > 0 and new_count % 50 == 0:
            _save_enriched(enriched, output_file)
            print(f"    --- Saved: {len(enriched)}/{len(refs)} ---")

        time.sleep(delay)

    # 5. Final save
    _save_enriched(enriched, output_file)

    # 6. Compute and print statistics
    hf_count = 0
    total_instances = 0
    for m in enriched.values():
        text = json.dumps(m, default=str).lower()
        if "huggingface" in text or "hugging face" in text:
            hf_count += 1
        total_instances += len(m.get("instances") or [])

    total_models = len(enriched)
    file_size = round(os.path.getsize(output_file) / (1024 * 1024), 1)

    print(f"\n{'='*55}")
    print(f"  Total models           : {total_models}")
    print(f"  Total instances        : {total_instances}")
    print(f"  From HuggingFace       : {hf_count} "
          f"({round(hf_count / max(total_models, 1) * 100, 2)}%)")
    print(f"  Non-HuggingFace        : {total_models - hf_count}")
    print(f"  Failed                 : {len(failed_refs)}")
    print(f"  File                   : {output_file} ({file_size} MB)")
    print(f"{'='*55}")

    if failed_refs:
        print(f"\n  Failed refs (first 10): {failed_refs[:10]}")

    return enriched


def _save_enriched(enriched_dict, filename):
    """
    Saves the enriched model cards dictionary to a JSON file.
    
    Parameters
    ----------
    enriched_dict : dict
        Dictionary mapping model ref (str) to model card data (dict).
    filename : str
        Output file path.
    """
    data = list(enriched_dict.values())
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)


---
## Step 5 - Trial Run (10 models)

Before running the full extraction (~2-3 hours), let's test with 10 models to make sure everything works.

**Expected time:** ~20 seconds


In [ ]:
# Trial run — only 10 models
enriched_trial = fetch_all_model_cards(limit=10)

Playground: To check if models have all the properties. Models that consist maximum properties are loaded here for reference.

In [ ]:
import json

api = KaggleApi()
api.authenticate()

detail = api.model_get("google/bert")

# Compare to_dict() vs manual extraction
d = detail.to_dict()
print("Keys from to_dict():")
print(list(d.keys()))
print(f"\nHas description: {bool(d.get('description'))}")
print(f"Has instances: {bool(d.get('instances'))}")

# Check if instances have all fields
if d.get("instances"):
    print(f"\nFirst instance keys: {list(d['instances'][0].keys())}")
    print(f"Has usage: {bool(d['instances'][0].get('usage'))}")

# Try to_json() instead
j = json.loads(detail.to_json())
print(f"\n\nKeys from to_json():")
print(list(j.keys()))
print(f"Has description: {bool(j.get('description'))}")

if j.get("instances"):
    print(f"First instance keys: {list(j['instances'][0].keys())}")
    print(f"Has usage: {bool(j['instances'][0].get('usage'))}")

In [ ]:
api = KaggleApi()
api.authenticate()

# Fetch 3 models fresh
test_refs = ["google/bert", "google/gemma", "tensorflow/efficientnet"]
results = []

for ref in test_refs:
    print(f"Fetching {ref} ...", end=" ", flush=True)
    detail = api.model_get(ref)
    results.append(detail.to_dict())
    print("OK")
    time.sleep(2)

# Save to a NEW file
with open("CLEAN_TEST_3_models.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

# Verify all fields are present
for m in results:
    print(f"\n{m['ref']}:")
    print(f"  description length : {len(m.get('description', ''))}")
    print(f"  instances          : {len(m.get('instances', []))}")
    print(f"  tags               : {m.get('tags')}")
    print(f"  publishTime        : {m.get('publishTime')}")
    print(f"  provenanceSources  : {m.get('provenanceSources')}")
    if m.get('instances'):
        inst = m['instances'][0]
        print(f"  1st instance usage : {len(inst.get('usage', ''))} chars")
        print(f"  1st instance license: {inst.get('licenseName')}")
        print(f"  1st instance framework: {inst.get('framework')}")

print(f"\nFile saved: CLEAN_TEST_3_models.json")

---
## Step 6 - Verify the Trial Output

Let's inspect the JSON to make sure we're getting the full model cards with all properties.


In [ ]:
# Load and inspect the trial output
with open("kaggle_models_full_cards_now.json", "r", encoding="utf-8") as f:
    trial_data = json.load(f)

print(f"Models saved: {len(trial_data)}")
print(f"\nKeys in first model: {list(trial_data[0].keys())}")

# Check a specific model
m = trial_data[0]
print(f"\n--- First Model ---")
print(f"  ref         : {m.get('ref')}")
print(f"  title       : {m.get('title')}")
print(f"  author      : {m.get('author')}")
print(f"  description : {str(m.get('description', ''))[:200]}...")
print(f"  instances   : {len(m.get('instances', []))}")
print(f"  tags        : {m.get('tags')}")
print(f"  provenance  : {m.get('provenance_sources')}")
print(f"  publish_time: {m.get('publish_time')}")
print(f"  vote_count  : {m.get('vote_count')}")

# Check an instance
if m.get("instances"):
    inst = m["instances"][0]
    print(f"\n--- First Instance ---")
    print(f"  slug        : {inst.get('slug')}")
    print(f"  framework   : {inst.get('framework')}")
    print(f"  license     : {inst.get('licenseName')}")
    print(f"  fineTunable : {inst.get('fineTunable')}")
    print(f"  trainingData: {inst.get('trainingData')}")
    print(f"  size (bytes): {inst.get('totalUncompressedBytes')}")


---
## Step 7 (Optional) - Full Extraction (All Models)

Now fetch the complete model card for all ~6,250 models.

**Expected time:** ~2-3 hours  
**Resumable:** If interrupted, just re-run this cell — it picks up where it left off.  
**Progress:** Saved every 50 models to `kaggle_models_full_cards.json`


In [ ]:
# Full run — all models
# If this was already partially run, it will resume automatically.
enriched = fetch_all_model_cards()


---
## Step 8 - Explore the Final Data

Load the complete dataset and explore what we've collected.


In [ ]:
import json

with open("kaggle_models_full_cards_now.json", "r", encoding="utf-8") as f:
    all_models = json.load(f)

print(f"Total models: {len(all_models)}")

# Count instances across all models
total_instances = sum(len(m.get("instances", [])) for m in all_models)
print(f"Total instances: {total_instances}")

# Unique authors
authors = set(m.get("author", "Unknown") for m in all_models)
print(f"Unique authors: {len(authors)}")

# Top 10 authors by model count
from collections import Counter
author_counts = Counter(m.get("author", "Unknown") for m in all_models)
print(f"\nTop 10 authors:")
for author, count in author_counts.most_common(10):
    print(f"  {author}: {count} models")

# Framework distribution across all instances
frameworks = []
for m in all_models:
    for inst in m.get("instances", []):
        frameworks.append(inst.get("framework", "unknown"))
fw_counts = Counter(frameworks)
print(f"\nFramework distribution:")
for fw, count in fw_counts.most_common():
    print(f"  {fw}: {count}")


---
## Step 9 - HuggingFace Model Analysis

Note: The actual HuggingFace models on Kaggle exist as a frontend integration layer that the API doesn't expose. In case, you found 'Hugging Face', this will confirm they're just mentioning HuggingFace somwhere in description, not sourced from it.

Identify which models come from or reference HuggingFace, and separate them.


In [ ]:
# Identify HuggingFace models by searching the entire model card
hf_models = []
non_hf_models = []

for m in all_models:
    text = json.dumps(m, default=str).lower()
    if "huggingface" in text or "hugging face" in text or "hugging-face" in text:
        hf_models.append(m)
    else:
        non_hf_models.append(m)

print(f"Total models       : {len(all_models)}")
print(f"HuggingFace models : {len(hf_models)} ({round(len(hf_models)/len(all_models)*100, 2)}%)")
print(f"Non-HuggingFace    : {len(non_hf_models)}")

# Show HuggingFace model details
if hf_models:
    print(f"\nHuggingFace models found:")
    for m in hf_models[:20]:
        print(f"  {m.get('ref'):50s}  by {m.get('author')}")

# Save non-HuggingFace models separately
with open("kaggle_models_no_hf.json", "w", encoding="utf-8") as f:
    json.dump(non_hf_models, f, indent=2, ensure_ascii=False, default=str)
print(f"\nNon-HuggingFace models saved to: kaggle_models_no_hf.json")



---
## Summary

This notebook extracted complete model card metadata from the Kaggle platform:

1. **Phase 1** — Listed all available models (~6,250) via the `model_list` API
2. **Phase 2** — Fetched full model cards for each model via `model_get`
3. **Analysis** — Identified and separated HuggingFace models

### Output Files
| File | Contents |
|------|----------|
| `kaggle_models_metadata.json` | Phase 1: Basic model list (refs, titles, authors) |
| `kaggle_models_full_cards.json` | Phase 2: Complete model cards with all metadata |
| `kaggle_models_no_hf.json` | Non-HuggingFace models only |

### Key Limitations
- Kaggle's API has a server-side pagination limit of ~6,250 models in the `model_list` endpoint
- For the complete catalog, Kaggle recommends using the [Meta Kaggle dataset](https://www.kaggle.com/datasets/kaggle/meta-kaggle) (but it lacks rich model card data)
- The `model_get` endpoint returns full data including descriptions, instances, and usage docs
